# TeCo classification sweep — one run, many setups

**CPU → Run all**, using **Zyion** for both Drive prompts. GPU is not needed. This is a broad CPU search; allow several hours (not yet benchmarked on your runtime). The notebook will save and resume every completed fit. Reopening the same revision reuses compact feature caches and completed cases.

**Data:** TeCo **task 1**, the original frozen positive/negative agent word-agreement selection, first stored fixation only. Same saved **40 Hz / 125 Hz** epochs; no source download, relabeling or new extraction. All readers meeting the previous ≥40/class rule remain in the cohort. No neutral class in this sweep.

## What changes
- **22 EEG input representations:** all-channel means in eight windows (0–200,100–200,200–300,224–304,300–500,500–800,0–500,0–800 ms); regional early/late/full time bins; six fixed regions at224–304ms; regional waveform statistics; regional2–30Hz log-band energy; GFP time bins; two training-only nuisance-residualized inputs. Band energy over0.8s is a coarse feature, not a precise spectral estimate.
- **Five model families / eight settings:** logistic regression C=0.01,0.1,1,10; shrinkage LDA; nonlinear RBF SVM; Extra Trees; small32-unit neural network (MLP). Previous CNNs were near chance; this notebook explores these other model families without another costly CNN search.
- **Exact averages of1,5,10,20,40 same-class trials.** Group within reader and partition; discard remainders. Larger sizes have fewer test examples; count them explicitly. No partial-group fallback.
- **Four separate evaluations:** unseen readers; unseen sentences; unseen word identities; mixed familiar-identity diagnostic. Mixed allows identities on both sides but never the same event. We measure actual overlap of identities rather than promise every identity is seen.

## How the search is evaluated
Three outer folds. Inside each training portion, two validation folds screen all22inputs with C1 logistic. The best three inputs advance to the eight model settings. Only the validation-selected winner is evaluated on the outer test fold, alongside a fixed EEG reference and **pre-onset, word/fixation-only and dummy controls**. We report separate scores for each averaging size and split; do not pick a grand winner from the test scores. This is roughly5,000small fits before coverage-related skips, not a full Cartesian search.

All imputation, scaling and nuisance regression fit training examples only. For every inner and outer split, purge training epochs within1000ms of test/validation events in the same reader/sentence. This is conservative for narrow windows and does not establish independence from all filtering effects. Record purged events and incomplete coverage. A condition needs all three outer folds before a pooled score is reported.

This uses data already explored in earlier experiments: the outer folds protect selection **within this run**, not a new independent dataset. Known sentiment labels construct averaged examples. Their accuracy is not individual-word prediction accuracy. Candidate development scores are not final results. No significance fishing or automated claims that the best percentage is a discovery.

**Storage:** at most **50 MB total** under `Thesis/Results/word_sentiment_eeg/teco_classification_sweep/`. This includes small per-reader feature caches, memberships, scores and plots. No new full EEG copy, no saved model bank, no existing files deleted. First run reads existing selected epoch shards; disconnects reuse completed readers and fits. Logs name reader X/12, split, fold, averaging size and candidate progress.


In [ ]:
# Runtime check: run before installation, Drive access, or analysis.
import os
import subprocess

REQUIRED_RUNTIME = 'CPU'
if os.environ.get('COLAB_TPU_ADDR') or os.environ.get('TPU_NAME'):
    raise RuntimeError(f'This notebook needs {REQUIRED_RUNTIME}, not TPU. Choose Runtime > Change runtime type, then rerun all.')
try:
    _gpu_probe = subprocess.run(
        ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
        capture_output=True, text=True, timeout=15, check=False)
except FileNotFoundError:
    _gpu_names = ''
except subprocess.TimeoutExpired:
    raise RuntimeError('GPU detection timed out. Reconnect the runtime and rerun this cell.') from None
else:
    if _gpu_probe.returncode:
        raise RuntimeError('GPU detection failed. Reconnect the runtime and rerun this cell.')
    _gpu_names = _gpu_probe.stdout.strip()
if REQUIRED_RUNTIME == 'GPU' and not _gpu_names:
    raise RuntimeError('GPU required. Choose Runtime > Change runtime type > T4 GPU, then Run all. No setup or analysis has started.')
if REQUIRED_RUNTIME == 'CPU' and _gpu_names:
    raise RuntimeError('CPU notebook: choose Runtime > Change runtime type > CPU (or hardware accelerator None), then Run all to avoid using GPU quota.')
print('Runtime verified:', _gpu_names if _gpu_names else 'CPU', flush=True)


In [ ]:
from google.colab import drive, auth
from pathlib import Path
import os, sys, subprocess, json
from IPython.display import display, Markdown, Image
import pandas as pd

drive.mount('/content/drive')
THESIS=Path('/content/drive/MyDrive/Thesis')
OUT=THESIS/'Results/word_sentiment_eeg/teco_classification_sweep'
REPO=Path('/content/word-sentiment-eeg-teco-sweep')
CODE_REVISION='befc90395e573f3d19b6bd6f2c7e81995181c415'
def git(*args):
    result=subprocess.run(['git','-c','credential.helper=','-c','http.extraheader=',*map(str,args)],
                          text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                          env=dict(os.environ,GIT_TERMINAL_PROMPT='0'))
    if result.returncode:
        raise RuntimeError('Git setup failed:\n'+result.stdout)
    if result.stdout:print(result.stdout,end='',flush=True)
if not REPO.exists():
    git('clone','--depth','1','https://github.com/parmisbathayan/word-sentiment-eeg.git',REPO)
else:
    if subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip():
        raise RuntimeError('Temporary checkout contains edits. Preserve them before updating.')
if subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()!=CODE_REVISION:
    git('-C',REPO,'fetch','--depth','1','origin',CODE_REVISION)
    fetched=subprocess.check_output(['git','-C',str(REPO),'rev-parse','FETCH_HEAD^{commit}'],text=True).strip()
    if fetched!=CODE_REVISION:raise RuntimeError('Fetched code differs from the pinned commit.')
    git('-C',REPO,'checkout','--detach','FETCH_HEAD')
if subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()!=CODE_REVISION:
    raise RuntimeError('Code checkout verification failed.')
# Temporary cloud-only scientific environment: no base-runtime upgrades/restart.
# BOOTSTRAP_START: also exercised verbatim in cloud CI.
# Use a file-rendering backend in subprocesses, not Colab's unavailable inline plugin.
os.environ['MPLBACKEND']='Agg'
ENV=Path('/content/word-eeg-paper-classification-python')
PYTHON=ENV/'bin/python'
def setup_command(command):
    print('SETUP:', ' '.join(map(str,command)), flush=True)
    result=subprocess.run(list(map(str,command)),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
    if result.stdout:print(result.stdout,flush=True)
    if result.returncode:
        raise RuntimeError('Scientific environment setup failed; the full diagnostic is printed above. No EEG processing started.')
# Colab may lack ensurepip. This also repairs a partially created environment.
setup_command([sys.executable,'-m','venv','--without-pip',str(ENV)])
setup_command([sys.executable,'-m','pip','--python',str(PYTHON),'install',
               '--only-binary=:all:','-r',str(REPO/'requirements-paper-classification.txt')])
setup_command([sys.executable,'-m','pip','--python',str(PYTHON),'check'])
setup_command([str(PYTHON),'-c',
    'import numpy,pandas,scipy,sklearn,matplotlib; '
    'print("Scientific environment ready:",numpy.__version__,pandas.__version__,scipy.__version__,sklearn.__version__,matplotlib.__version__)'])
# BOOTSTRAP_END
print('Code revision:',subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip())

# Verify account quota before loading any EEG. Use Zyion for both prompts.
auth.authenticate_user()
import google.auth
from google.auth.transport.requests import AuthorizedSession
credentials,_=google.auth.default()
response=AuthorizedSession(credentials).get('https://www.googleapis.com/drive/v3/about',
    params={'fields':'storageQuota,user(emailAddress)'},timeout=30)
response.raise_for_status();about=response.json();quota=about['storageQuota']
email=about.get('user',{}).get('emailAddress','unavailable')
print('Authenticated quota account:',email)
if email!='zyionygvmutch39@gmail.com':
    raise RuntimeError('Choose your Zyion account for Drive authentication and mounting; it contains the saved trials.')
if 'limit' not in quota:
    raise RuntimeError('Drive did not expose a storage limit; cannot verify free space.')
FREE_DRIVE_BYTES=max(0,int(quota['limit'])-int(quota.get('usage',0)))
used=sum(p.stat().st_size for p in OUT.rglob('*') if p.is_file()) if OUT.exists() else 0
if FREE_DRIVE_BYTES<max(0,50_000_000-used):
    raise RuntimeError('Reserve at most 50 MB for results. This notebook reads only saved TeCo epochs once for compact feature vectors; no new extraction.')
print(f'Free space: {FREE_DRIVE_BYTES/1e9:.2f} GB; new output cap: 50 MB.')
print('New output folder:',OUT)


In [ ]:
sys.path.insert(0,str(REPO/'scripts'))
from colab_progress import run_with_progress
command=[str(PYTHON),'-u',str(REPO/'scripts/teco_classification_sweep.py'),'--thesis-root',str(THESIS),'--free-drive-bytes',str(FREE_DRIVE_BYTES)]
run_with_progress(command,OUT/'runtime_status.json',activity='TeCo nested input/model/averaging sweep')


In [ ]:
import hashlib
from IPython.display import Image, display, Markdown
state=json.loads((OUT/'complete.json').read_text())
assert state['status']=='completed'
for name,sha in state['checksums'].items():
    if hashlib.sha256((OUT/name).read_bytes()).hexdigest()!=sha:raise RuntimeError('Saved output checksum failed: '+name)
scores=pd.read_csv(OUT/'summary.csv')
display(Markdown('## Finished and verified — outer-test results'))
display(scores.round(3))
display(Image(filename=str(OUT/'sweep.png')))


In [ ]:
results=json.loads((OUT/'outer_results.json').read_text())
chosen=pd.DataFrame([{k:r.get(k) for k in ['mode','fold','size','variant','input','model','status','balanced_accuracy','test_examples','test_trials','reason']} for r in results])
display(Markdown('## Which input/model was selected in each training fold?'))
display(chosen[chosen.variant.eq('selected_eeg')].round(3))
display(Markdown('## Coverage: larger averages leave fewer test examples'))
display(scores[scores.variant.eq('selected_eeg')].round(3))
print('Files:',OUT)
print('summary.csv = final scores; outer_results.json = predictions and chosen models; development_scores.csv = validation search; fits/ = resumable checkpoints; features/ = reusable small vectors.')

development=pd.read_csv(OUT/'development_scores.csv')
if 'reason' in development.columns:
    missing=development[development.status.eq('unavailable')]
    if len(missing):
        display(Markdown('## Unavailable conditions — no silent smaller-average fallback'))
        display(missing[[k for k in ['mode','fold','size','reason'] if k in missing.columns]].drop_duplicates())
